In [ ]:
# Resolve project inputs consistently when Jupyter opens this subdirectory.
import os

from paths import ROOT

os.chdir(ROOT)
# Set figure styles and output paths for the DPX switch-neuron comparison.
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

color_pal_cueprobes = {
    "AX": "#F7AD57",
    "AY": "#F1565C",
    "BX": "#AA5DA5",
    "BY": "#5598CA",
}


figdir = Path("artifacts/dpx/figures/")
figdir.mkdir(parents=True, exist_ok=True)

### read data

In [ ]:
# Load the preserved monkey and model population summaries used by the manuscript.
neural_activity2plot_mk = pickle.load(open("artifacts/dpx/monkey/population.pkl", "rb"))
neural_activity2plot_md = pickle.load(open("artifacts/dpx/predictive/population.pkl", "rb"))

In [ ]:
# Compute cue-period B minus A and probe-period AY minus AX contrasts.
def get_diff_info(neural_activity2plot_used, n_ids):
    dt = neural_activity2plot_used["dt"]
    cue_start = 500 // dt
    cue_end = 1500 // dt
    probe_start = 2500 // dt
    probe_end = 3000 // dt
    n_activity_pre_B = (
        neural_activity2plot_used["pre"]["BX"][:, n_ids]
        + neural_activity2plot_used["pre"]["BY"][:, n_ids]
    ) / 2
    n_activity_pre_A = (
        neural_activity2plot_used["pre"]["AX"][:, n_ids]
        + neural_activity2plot_used["pre"]["AY"][:, n_ids]
    ) / 2
    diff_BA_pre = (n_activity_pre_B - n_activity_pre_A)[cue_start:cue_end].mean(axis=-1)

    data_diff = []
    for d in diff_BA_pre:
        data_diff.append(
            [
                float(d),
                "B vs A",
                "Prepotent",
            ]
        )

    n_activity_bal_B = (
        neural_activity2plot_used["bal"]["BX"][:, n_ids]
        + neural_activity2plot_used["bal"]["BY"][:, n_ids]
    ) / 2
    n_activity_bal_A = (
        neural_activity2plot_used["bal"]["AX"][:, n_ids]
        + neural_activity2plot_used["bal"]["AY"][:, n_ids]
    ) / 2

    diff_BA_bal = (n_activity_bal_B - n_activity_bal_A)[cue_start:cue_end].mean(axis=-1)

    for d in diff_BA_bal:
        data_diff.append(
            [
                float(d),
                "B vs A",
                "Balanced",
            ]
        )

    diff_AYAX_pre = (
        neural_activity2plot_used["pre"]["AY"][:, n_ids]
        - neural_activity2plot_used["pre"]["AX"][:, n_ids]
    )[probe_start:probe_end].mean(axis=-1)

    for d in diff_AYAX_pre:
        data_diff.append(
            [
                float(d),
                "AY vs AX",
                "Prepotent",
            ]
        )

    diff_AYAX_bal = (
        neural_activity2plot_used["bal"]["AY"][:, n_ids]
        - neural_activity2plot_used["bal"]["AX"][:, n_ids]
    )[probe_start:probe_end].mean(axis=-1)

    for d in diff_AYAX_bal:
        data_diff.append(
            [
                float(d),
                "AY vs AX",
                "Balanced",
            ]
        )

    df_diff = pd.DataFrame(data_diff, columns=["diff", "type", "comparison"])
    return df_diff

In [ ]:
# Plot the selected switch populations and their condition contrasts.
fig, axes = plt.subplots(
    2,
    3,
    figsize=(8, 3),
    sharex="col",
)
axes[0, 0].sharey(axes[0, 1])
axes[1, 0].sharey(axes[1, 1])

axes[0, 0].sharex(axes[1, 0])
axes[0, 1].sharex(axes[1, 1])

axes[0, 2].sharex(axes[1, 2])

# axes[0, 0].set_ylabel("Monkey")
# axes[1, 0].set_ylabel("Model")

for cue_probe in ["AX", "AY", "BX", "BY"]:
    maxt = 3200
    dt = neural_activity2plot_mk["dt"]
    n_ids = neural_activity2plot_mk["ids2plot"]
    n_activity_bal = neural_activity2plot_mk["bal"][cue_probe][:, n_ids][: maxt // dt]
    n_activity_pre = neural_activity2plot_mk["pre"][cue_probe][:, n_ids][: maxt // dt]

    time_steps = np.arange(n_activity_bal.shape[0]) * dt

    ax = axes[0, 0]
    ax.plot(
        time_steps,
        n_activity_bal.mean(axis=-1),
        color=color_pal_cueprobes[cue_probe],
        label=cue_probe,
    )
    ax = axes[0, 1]
    ax.plot(
        time_steps,
        n_activity_pre.mean(axis=-1),
        color=color_pal_cueprobes[cue_probe],
        label=cue_probe,
    )

    # model
    dt = neural_activity2plot_md["dt"]

    n_ids = neural_activity2plot_md["ids2plot"]
    n_activity_bal = neural_activity2plot_md["bal"][cue_probe][:, n_ids][: maxt // dt]
    n_activity_pre = neural_activity2plot_md["pre"][cue_probe][:, n_ids][: maxt // dt]
    time_steps = np.arange(n_activity_bal.shape[0]) * dt

    ax = axes[1, 0]
    ax.plot(
        time_steps,
        n_activity_bal.mean(axis=-1),
        color=color_pal_cueprobes[cue_probe],
        label=cue_probe,
    )
    ax.set_xlabel("Time (ms)")
    ax = axes[1, 1]
    ax.plot(
        time_steps,
        n_activity_pre.mean(axis=-1),
        color=color_pal_cueprobes[cue_probe],
        label=cue_probe,
    )
    ax.set_xlabel("Time (ms)")


for ax in [axes[0, 0], axes[1, 0], axes[0, 1], axes[1, 1]]:
    ax.axvspan(500, 1500, color="silver", alpha=0.2)
    ax.axvspan(2500, 3000, color="silver", alpha=0.2)
    # ax.legend()
    if ax is axes[1, 1]:
        ax.legend(
            loc="upper left",
            #   bbox_to_anchor=(1.1, 1.1),
            frameon=False,
        )


df_diff = get_diff_info(neural_activity2plot_mk, neural_activity2plot_mk["ids2plot"])
sns.barplot(data=df_diff, x="type", y="diff", hue="comparison", ax=axes[0, 2], palette="Paired")
axes[0, 2].set_ylim(0, df_diff["diff"].max())

df_diff = get_diff_info(neural_activity2plot_md, neural_activity2plot_md["ids2plot"])
sns.barplot(data=df_diff, x="type", y="diff", hue="comparison", ax=axes[1, 2], palette="Paired")

axes[1, 2].set_ylim(0, df_diff["diff"].max())
axes[1, 2].legend_.set_title(None)
axes[1, 2].legend_.set_frame_on(False)
axes[0, 2].legend_.remove()

plt.tight_layout()
sns.despine()
fig.savefig(figdir / "fig3.png", dpi=300)
plt.show()